In [6]:
"""
Chord Alphabet Reduction
========================
Reduces the chord vocabulary in integer-indexing-key.csv to a maximum of
7th chords with no inversions.
 
Each entry in the CSV is a SALAMI-style chord label of the form:
    <pitch_class>:<quality>[/<bass>]
 
where:
  - pitch_class : int 0–11, semitones above the tonic (0 = tonic/C if tonic is C)
  - quality     : chord quality string, possibly with extensions in parentheses
  - /bass       : optional bass note indicating an inversion (semitones above tonic)
 
The output is a mapping from every original integer key to a reduced integer key,
where the reduced key corresponds to the simplified chord (7th chord max, root position).
 
Target chord qualities after reduction (the reduced alphabet):
    Triads : maj, min, dim, aug, sus4
    7ths   : maj7, min7, 7 (dominant), dim7, hdim7, minmaj7
 
Design principles:
  - Strip inversions first (remove "/<bass>").
  - Strip extensions beyond the 7th (9ths, 11ths, 13ths, altered notes
    in parentheses) while preserving the core chord type.
  - sus4(b7, ...) → dominant 7 (the b7 is structurally present; sus4 is colour).
  - sus2 → sus4 (closest stable triad substitute).
  - Power chords (:5) and bare roots (:1) → maj (neutral default).
  - Added-note chords (maj6, maj(9), etc.) strip to their triad base.
  - If a reduced chord already exists in the original vocabulary, reuse its key.
  - If not, assign a new key (> max existing key).
"""

import csv
import re
from pathlib import Path

## Constants
Target quality normalization table
Explicit map from quality strings found in the CSV to their reduced form.
 

In [7]:

QUALITY_MAP: dict[str, str] = {
    # --- Target triads ---
    "maj":      "maj",  # major triad = root + maj3 + P5 (e.g. C-E-G)
    "min":      "min",  # minor triad = root + min3 + P5 (e.g. C-Eb-G)
    "dim":      "dim",  # diminished triad = root + min3 + dim5 (e.g. C-Eb-Gb)
    "aug":      "aug",  # augmented triad = root + maj3 + aug5 (e.g. C-E-G#)
    "sus4":     "sus4", # suspended 4th = root + P4 + P5; the 3rd is replaced by the 4th (e.g. C-F-G)
    # no standard 7th-chord equivalent, mapped to sus4 as the closest stable triad
    "sus2":     "sus2", # suspended 2nd = root + M2 + P5; the 3rd is replaced by the 2nd (e.g. C-D-G);
                        
 
    # --- Target 7th chords ---
    "7":        "7",       # dominant 7th = maj triad + min7 (e.g. C-E-G-Bb)
    "maj7":     "maj7",    # major 7th = maj triad + maj7 (e.g. C-E-G-B)
    "min7":     "min7",    # minor 7th = min triad + min7 (e.g. C-Eb-G-Bb)
    "dim7":     "dim7",    # fully diminished 7th = dim triad + dim7 (e.g. C-Eb-Gb-Bbb)
    "hdim7":    "hdim7",   # half-diminished (ø7) = dim triad + min7 (e.g. C-Eb-Gb-Bb)
    "minmaj7":  "minmaj7", # minor-major 7th = min triad + maj7 (e.g. C-Eb-G-B) 
 
    # --- Suspended 7ths ---
    # the b7 is the structurally defining tone, sus4 is a colour/suspension.
    "sus4(b7)":         "sus4(b7)", # Mixolydian sus: root + P4 + P5 + min7 (e.g. C-F-G-Bb)
    "sus4(b7,9)":       "sus4(b7)", # adds maj9 on top; the 9th is a colour extension beyond our target
    "sus4(b7,9,13)":    "sus4(b7)", # further adds maj13; both 9 and 13 are stripped
    "sus4(b7,9,#11)":   "sus4(b7)", # Lydian-dominant colouring (#11 = raised tritone substitution); stripped
    "sus2(b7)":         "sus2(b7)", # same dominant function with M2 replacing the 3rd instead of P4
 
    # --- Dominant extensions (strip to dominant 7th) ---
    "9":        "7",   # dominant 9th  = 7 + 9
    "11":       "7",   # dominant 11th = 7 + 9 + 11
    "13":       "7",   # dominant 13th = 7 + 9 + 11 + 13
 
    # --- Major extensions (strip to maj7) ---
    "maj9":          "maj7",
    "maj9(#11)":     "maj7",
    "maj11":         "maj7",
    "maj13":         "maj7",
    "maj7(#11)":     "maj7",
    "maj7(#5)":      "maj7",
    "maj7(#9)":      "maj7",
 
    # --- Minor extensions (strip to min7) ---
    "min9":      "min7",
    "min9(b13)": "min7",
    "min11":     "min7",
    "min13":     "min7",
 
    # --- Added-note chords: strip the added note, keep the triad ---
    # These are NOT 7th chords; the 6th/added 9th is a colour tone, not the 7th.
    "maj6":      "maj",
    "maj6(9)":   "maj",
    "maj(9)":    "maj",
    "maj(11)":   "maj",
    "min6":      "min",
    "min(9)":    "min",
    "min(11)":   "min",
 
    # --- Power chord / bare root: neutral fallback to maj; mode cannot be inferred ---
    "5":        "maj",
    "1":        "maj",
 
    # --- augmented 7th variants ---
    # the aug5 is a chromatic colour tone; the maj7 character is retained
    "augmaj7":  "maj7",  # aug triad + maj7 (e.g. C-E-G#-B); 
}

# ---------------------------------------------------------------------------
# Regex helpers
# ---------------------------------------------------------------------------
 
# Inversion: everything from the last "/" to end-of-string, e.g. "/5", "/b7"
_INVERSION_RE = re.compile(r"/[^/]+$")
 
# Parenthesised alterations/extensions, e.g. "(b7,9)", "(#11)", "(9)"
_PARENS_RE = re.compile(r"\([^)]*\)")
 
# Numeric extension suffixes standing alone after the base quality: 9, 11, 13
# Only matches when they appear at the END of the string (after parens removed)
_EXT_SUFFIX_RE = re.compile(r"(9|11|13)$")
 
# Detects b7 inside any parenthesised group — used to identify dominant-7th
# sus chords before we strip the parentheses
_B7_IN_PARENS_RE = re.compile(r"\([^)]*b7[^)]*\)")

## Chord Reduction Helper Functions

In [8]:

def reduce_quality(quality_full: str) -> str:
    """
    Reduce a chord quality string to a triad or 7th chord quality.
 
    Parameters
    ----------
    quality_full : str
        Full quality string from the CSV, e.g. "maj/5", "sus4(b7,9)",
        "min7/b7", "maj9", "5", "hdim7".
 
    Returns
    -------
    str
        One of the target qualities: maj, min, dim, aug, sus4,
        7, maj7, min7, dim7, hdim7, minmaj7.
    """
    # Step 1: Strip inversion (the "/bass" suffix).
    # This must happen before any other step since the "/" only separates
    # quality from bass note, not within the quality itself.
    q = _INVERSION_RE.sub("", quality_full).strip()
 
    # Step 2: Exact lookup in the normalization table (most common path).
    if q in QUALITY_MAP:
        return QUALITY_MAP[q]
 
    # Step 3: Detect sus + b7 before stripping parens.
    # e.g. "sus4(b7,9,13)" — the b7 makes it a dominant 7th regardless of
    # how many extra colour tones are present.
    if re.match(r"sus[24]", q) and _B7_IN_PARENS_RE.search(q):
        return "7"
 
    # Step 4: Strip parenthesised content and retry.
    q_stripped = _PARENS_RE.sub("", q).strip()
 
    if q_stripped in QUALITY_MAP:
        return QUALITY_MAP[q_stripped]
 
    # Step 5: Strip trailing numeric extension suffixes (9, 11, 13) and retry.
    q_base = _EXT_SUFFIX_RE.sub("", q_stripped).strip()
 
    if q_base in QUALITY_MAP:
        return QUALITY_MAP[q_base]
 
    # Step 6: Prefix match against target qualities, longest prefix first.
    # This handles unusual compound strings not caught above.
    for prefix in ("minmaj7", "hdim7", "augmaj7", "maj7", "min7", "dim7",
                   "maj6", "min6", "aug", "dim", "min", "maj",
                   "sus4", "sus2", "7", "5", "1"):
        if q_base.startswith(prefix):
            return QUALITY_MAP.get(prefix, prefix)
 
    # Step 7: Final fallback — if we truly cannot identify the quality, use maj.
    # Log a warning so developers can extend the table if needed.
    print(f"  [WARNING] Unrecognised quality: {quality_full!r} → defaulting to 'maj'")
    return "maj"
 
 
def reduce_chord_label(label: str) -> str:
    """
    Reduce a full SALAMI chord label (e.g. "5:maj/5", "7:sus4(b7,9)")
    to its simplified form (e.g. "5:maj", "7:7").
 
    Parameters
    ----------
    label : str
        A chord label of the form "<pitch_class>:<quality>[/<bass>]".
 
    Returns
    -------
    str
        Reduced chord label with no inversion and at most a 7th chord quality.
 
    Raises
    ------
    ValueError
        If the label does not contain a ":" separator.
    """
    if ":" not in label:
        raise ValueError(f"Chord label must contain ':'; got {label!r}")
 
    colon = label.index(":")
    pc      = label[:colon]           # e.g. "0", "10"
    quality = label[colon + 1:]       # e.g. "maj/5", "sus4(b7,9)"
 
    reduced_quality = reduce_quality(quality)
    return f"{pc}:{reduced_quality}"
 
 

## Building mappings

In [10]:

# ---------------------------------------------------------------------------
# Build reduced chord code mapping
# ---------------------------------------------------------------------------

def load_csv(path: str | Path) -> dict[int, str]:
    """
    Load integer-indexing-key.csv into a dict mapping integer key → chord label.
 
    Parameters
    ----------
    path : str or Path
        Path to the CSV file (expected columns: "key", "value").
 
    Returns
    -------
    dict[int, str]
        e.g. {1: "0:maj", 2: "5:maj", 3: "7:maj", ...}
    """
    result: dict[int, str] = {}
    with open(path, newline="", encoding="utf-8") as fh:
        reader = csv.DictReader(fh)
        for row in reader:
            result[int(row["key"])] = row["value"].strip()
    return result
 
 
def build_reduction_map(original: dict[int, str]) -> dict[int, int]:
    """
    Reduce every chord in the original vocabulary and assign a new ordinal
    reduced code ranging from 1 to N (where N is the number of unique reduced
    chord labels). The ordinal codes are assigned in the order that each new
    unique reduced label is first encountered, iterating over the original
    keys in ascending order.
 
    Parameters
    ----------
    original : dict[int, str]
        Original key → chord label mapping from the CSV.
 
    Returns
    -------
    list[dict]
        One dict per original entry, sorted by original key, with fields:
          - original_code  : int   (the original integer key from the CSV)
          - original_label : str   (the original chord label, e.g. "5:maj/5")
          - reduced_label  : str   (the simplified label, e.g. "5:maj")
          - reduced_code   : int   (new ordinal integer 1..N)
    """
    # Assign ordinal reduced codes in first-encountered order (by original_code asc).
    # This ensures deterministic, reproducible output.
    ordinal_map: dict[str, int] = {}   # reduced_label → ordinal reduced_code
    next_ordinal = 1
 
    rows = []
    for orig_key, orig_label in sorted(original.items()):
        reduced_label = reduce_chord_label(orig_label)
 
        if reduced_label not in ordinal_map:
            ordinal_map[reduced_label] = next_ordinal
            next_ordinal += 1
 
        rows.append({
            "original_code":  orig_key,
            "original_label": orig_label,
            "reduced_label":  reduced_label,
            "reduced_code":   ordinal_map[reduced_label],
        })
 
    return rows
 
 
def save_reduction_csv(rows: list[dict], out_path: str | Path) -> None:
    """
    Write the reduction table to a CSV file.
 
    Columns: original_code, original_label, reduced_label, reduced_code
 
    Parameters
    ----------
    rows : list[dict]
        Output of build_ordinal_reduction().
    out_path : str or Path
        Destination file path.
    """
    fieldnames = ["original_code", "original_label", "reduced_label", "reduced_code"]
    with open(out_path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"Saved {len(rows)} rows → {out_path}")
 

In [11]:
import sys

# Path to the CSV — adjust if needed
csv_path = Path("./integer-indexing-key.csv")
original = load_csv(csv_path)
# Build the full reduction table with ordinal reduced codes (1..N)
rows = build_reduction_map(original)
out_path = csv_path.parent / "reduced_chord_codebook.csv"
save_reduction_csv(rows, out_path)

# Print a brief summary
n_unique_reduced = len({r["reduced_code"] for r in rows})
n_changed   = sum(1 for r in rows if r["original_label"] != r["reduced_label"])
n_unchanged = len(rows) - n_changed
print(f"Original vocabulary : {len(original)} chords")
print(f"Reduced vocabulary  : {n_unique_reduced} unique chords")
print(f"Reduced codes range : 1 – {n_unique_reduced}")
print(f"Chords changed      : {n_changed}")
print(f"Chords unchanged    : {n_unchanged}")
print()

Saved 692 rows → reduced_chord_codebook.csv
Original vocabulary : 692 chords
Reduced vocabulary  : 137 unique chords
Reduced codes range : 1 – 137
Chords changed      : 564
Chords unchanged    : 128

